***

Preparing Workspace

***

In [ ]:


EXPORT=False

## Indicators:
# Cost_4


import pandas as pd
from pathlib import Path
import plotly.express as px


PATH_GIT = Path.cwd().parent.parent
PATH_CONFIG0 = PATH_GIT / 'config'

# SharePoint OneDrive paths
PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
PATH_MAIN = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_ORIG = Path(r'I:\Projects\Josh\Regional Monitoring\Task 9. Collect new data\FFIEC')
FILE_ABOUT = PATH_SP / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'
FILE_HOUSING = PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset' / 'HCD_SACOG_Summary_with_MPO.xlsx' # Developed by Warren


import sys
sys.path.append(str(PATH_CONFIG0))
import functions as func

def export_housing(df, indicator, source, sample_type, geography, year_start, year_end, path_out):   
    workbook_name = f'{indicator} {geography}_{source}.xlsx'
    path_out_workbook = path_out / workbook_name
    if indicator in ['Cost_4']:
        if geography == 'MPO':
            geography = 'Six-County Sacramento Region'
    df_about = func.write_about(sample_type    = sample_type
                                , indicator    = indicator
                                , year_start   = year_start
                                , year_end     = year_end
                                , geography    = geography)
    if indicator in ['Cost_4']:
        if geography == 'Six-County Sacramento Region':
            geography = 'MPO'
    with pd.ExcelWriter(path_out_workbook, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
        df      .to_excel(writer, sheet_name=geography, index=False,             )

        


In [ ]:


year_start = 2018
year_end   = 2024

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(FILE_HOUSING, sheet_name=str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df, ignore_index=True)
df_housing = df_housing[~df_housing['JURIS_NAME'].str.contains('https')]
df_housing.loc[df_housing['JURIS_NAME'].str.contains('COUNTY'), 'JURIS_NAME'] = 'UNINCORPORATED'

df_housing.head()



In [ ]:


gb_jurisdiction = df_housing.groupby(['CNTY_NAME', 'JURIS_NAME', 'Year'], as_index=False)
gb_county       = df_housing.groupby(['CNTY_NAME',               'Year'], as_index=False)
gb_mpo          = df_housing.groupby([                           'Year'], as_index=False)

metrics = ['Total', 'CO_VLI', 'CO_LI', 'CO_MI', 'CO_AMI']

#Jurisdiction level
print('Organizing indicator Cost_4 by Jurisdictions')
df_cost4_a = gb_jurisdiction[metrics].sum() 
display(df_cost4_a.head(5))

#County level 
print('Organizing indicator Cost_4 by Counties')
df_cost4_b = gb_county[metrics].sum()
display(df_cost4_b.head(5))

#MPO level 
print('Organizing indicator Cost_4 by MPO')
df_cost4_c = gb_mpo[metrics].sum()
display(df_cost4_c.head(5))


# Plotting
df_plot = df_cost4_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year']) 
df_plot = df_plot[df_plot['variable'] != 'Total']
df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['year'])['value'].transform('sum')
df_plot = df_plot.sort_values(['year', 'variable'], ascending = [False, True])
display(df_plot.head())


fig = px.line(df_plot, x='year', y='percentage', color='variable', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

fig.show()



In [ ]:


if EXPORT:

    indicator = 'Cost_4'
    source = 'SACOG HCD Summarized Data'
    sample_type = 'SACOG Housing'

    # Update overall about documentations workbook
    df_about = func.write_about(sample_type  = sample_type
                                , indicator  = indicator
                                , year_start = year_start
                                , year_end   = year_end)
    with pd.ExcelWriter(FILE_ABOUT, mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
        df_about.to_excel(writer, index = False, sheet_name = indicator, header = False)


    print('Exporting...');print()
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Cost' / f'{indicator} RHNA Income', PATH_SERVER]

    for path_out in paths_out:
        
        geography = 'Jurisdictions'
        export_housing(df_cost4_a, indicator, source, sample_type, geography, year_start, year_end, path_out)
            
        geography = 'Counties'
        export_housing(df_cost4_b, indicator, source, sample_type, geography, year_start, year_end, path_out)
        
        geography = 'MPO'
        export_housing(df_cost4_c, indicator, source, sample_type, geography, year_start, year_end, path_out)

    print('Success!!')

    